# CRSP Data Pull from WRDS

**Root cause of your error:** `dlret` does NOT exist in `crsp.dsf`.  
Delisting returns live in a separate table: `crsp.dsedelist`.  

This notebook pulls the three relevant tables separately, then merges them in pandas.

In [2]:
import wrds
import pandas as pd
import numpy as np
from pathlib import Path

db = wrds.Connection()

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


## Step 1 – Daily stock file + security characteristics (one SQL query)

`crsp.dsf` has: `permno, date, ret, prc, shrout`  
`crsp.dsenames` has: `shrcd, exchcd, siccd` (with effective date windows)

In [3]:
df = db.raw_sql("""
    SELECT
        a.permno,
        a.date,
        a.ret,
        a.prc,
        a.shrout,
        b.shrcd,
        b.exchcd,
        b.siccd
    FROM crsp.dsf AS a
    LEFT JOIN crsp.dsenames AS b
        ON  a.permno   = b.permno
        AND b.namedt  <= a.date
        AND a.date    <= b.nameendt
    WHERE a.date BETWEEN '1998-01-01' AND '2025-12-31'
""", date_cols=['date'])

print(f"dsf + names shape: {df.shape}")
df.head(3)

dsf + names shape: (51775326, 8)


,permno,date,ret,prc,shrout,shrcd,exchcd,siccd
0,10066,1998-04-06,<NA>,<NA>,59095.0,11,0,3573
1,10066,1998-04-07,<NA>,<NA>,59095.0,11,0,3573
2,10066,1998-04-08,<NA>,<NA>,59095.0,11,0,3573


## Step 2 – Delisting returns (separate table)

`crsp.dsedelist` columns that matter: `permno, dlstdt, dlret`

In [4]:
dsedelist = db.raw_sql("""
    SELECT permno, dlstdt, dlret
    FROM crsp.dsedelist
    WHERE dlstdt BETWEEN '1998-01-01' AND '2025-12-31'
""", date_cols=['dlstdt'])

print(f"dsedelist shape:        {dsedelist.shape}")
print(f"dlret non-null rows:    {dsedelist['dlret'].notna().sum()}")
dsedelist.head(3)

dsedelist shape:        (25777, 3)
dlret non-null rows:    15172


,permno,dlstdt,dlret
0,10001,2017-08-03,0.0
1,10002,2013-02-15,0.010906
2,10009,2000-11-03,0.003774


## Step 3 – Merge delisting returns onto the main panel

Match `(permno, dlstdt)` ↔ `(permno, date)`.  
For most rows `dlret` will be NaN — that is correct and expected.

In [5]:
# dlstdt is the delisting date; rename to 'date' so we can merge on it
dsedelist_merge = dsedelist.rename(columns={'dlstdt': 'date'})

df = df.merge(
    dsedelist_merge[['permno', 'date', 'dlret']],
    on=['permno', 'date'],
    how='left'
)

print(f"Final shape:             {df.shape}")
print(f"dlret non-null rows:     {df['dlret'].notna().sum()}")
print(f"Columns:                 {df.columns.tolist()}")

Final shape:             (51775326, 9)
dlret non-null rows:     15172
Columns:                 ['permno', 'date', 'ret', 'prc', 'shrout', 'shrcd', 'exchcd', 'siccd', 'dlret']


## Step 4 – Rename to match pipeline's column_mapping.yaml

In [ ]:
df = df.rename(columns={
    'permno': 'PERMNO',
    'ret':    'RET',
    'dlret':  'DLRET',
    'prc':    'PRC',
    'shrout': 'SHROUT',
    'shrcd':  'SHRCD',
    'exchcd': 'EXCHCD',
    'siccd':  'SICCD',
    # 'date' stays lowercase — matches the mapping
})

print("Final columns:", df.columns.tolist())
print("\nNull counts:")
print(df.isnull().sum())
print("\nDate range:", df['date'].min(), "—", df['date'].max())
print("Unique PERMNOs:", df['PERMNO'].nunique())

Final columns: ['PERMNO', 'date', 'RET', 'PRC', 'SHROUT', 'SHRCD', 'EXCHCD', 'SICCD', 'DLRET']

Null counts:
PERMNO           0
date             0
RET         702851
PRC         685526
SHROUT           1
SHRCD            0
EXCHCD           0
SICCD            0
DLRET     51760154
dtype: int64

Date range: 1998-01-02 00:00:00 — 2024-12-31 00:00:00
Unique PERMNOs: 25777


## Step 5 – Save to parquet

In [7]:
out_path = Path('data/raw/crsp_daily.parquet')
out_path.parent.mkdir(parents=True, exist_ok=True)

df.to_parquet(out_path, index=False)
print(f"Saved {len(df):,} rows  →  {out_path}")
print(f"File size: {out_path.stat().st_size / 1e6:.1f} MB")

Saved 51,775,326 rows  →  data/raw/crsp_daily.parquet
File size: 401.8 MB


## Step 6 – Verify the pipeline cleaning step works

In [8]:
import sys
sys.path.insert(0, '.')

from src.data_cleaning import load_raw_crsp, clean_crsp_data

raw    = load_raw_crsp(out_path)
cleaned = clean_crsp_data(raw)

print(f"Cleaned: {len(cleaned):,} rows | {cleaned['permno'].nunique():,} securities")
print(f"Date range: {cleaned['date'].min().date()} — {cleaned['date'].max().date()}")
print(cleaned[['date','permno','ret_total','me']].head())

Cleaned: 30,703,331 rows | 14,706 securities
Date range: 1998-01-02 — 2024-12-31
        date  permno  ret_total         me
0 1998-01-02   10001   0.000000    21555.0
1 1998-01-02   10002   0.081633   112519.0
2 1998-01-02   10009   0.029126    63308.5
3 1998-01-02   10011   0.000000   92186.25
4 1998-01-02   10012   0.000000  80942.125


In [3]:
import numpy as np

# .npz 파일 로드
with np.load('data/processed/covariance_pairs_test.npz') as data:
    # 저장된 배열의 키 확인
    print(data.files)  # 출력: ['key1', 'key2']
    
    # 키를 이용해 배열 추출
    loaded_array1 = data['target_raw']
    
    print(loaded_array1.shape)


['condition_raw', 'target_raw', 'condition_vech', 'target_vech', 'condition_scaled', 'target_scaled']
(1434, 10, 10)
